# Example climb: nonlinear feature interactions

## 1. Notebook setup

### 1.1. Imports

In [ ]:
# Standard library imports
import pickle
from itertools import combinations

# Third party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.stats import pearsonr
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from hill_climber import HillClimber

### 1.2. Run configuration

In [ ]:
# Set True to re-run optimization, False to load checkpoint from disk
run_optimization = True

# Input dataset size
n = 1000

# Hill climber parameters
max_time=10               # Maximum time (in minutes) to run optimization
n_replicas=20             # Number of replicas (workers) in parallel tempering
initial_step_spread=0.5   # Fraction of input range to sample initially
final_step_spread=0.01    # Fraction of input range to sample finally
step_spread_scheme='zeno' # Cooling scheme for step spread
exchange_interval=1000    # Steps between replica exchange attempts
T_max_initial=100         # Replica initial max temperature
T_min_initial=1e-2        # Replica initial min temperature
T_max_final=0             # Replica final max temperature
T_min_final=0             # Replica final min temperature

# Filename for results database
db_path='../data/feature_interactions.db' # Path to results database

# Filename for results checkpoint
output_file = '../data/feature_interaction_results.pkl'

# Set random seed for reproducibility
np.random.seed(315)

### 1.3. Input data

In [ ]:
# Create input distributions for 3 features (x1, x2, x3) and 1 label (y)
data = pd.DataFrame({
    'x1': np.random.uniform(-1, 1, n),
    'x2': np.random.uniform(-1, 1, n),
    'x3': np.random.uniform(-1, 1, n),
    'y': np.random.uniform(-1, 1, n)
})

data.head()

#### Descriptive statistics

In [ ]:
print(data.describe())

#### Individual feature-label Pearson correlation coefficients

In [ ]:
# Calculate Pearson correlation coefficients between each feature and the label
for col in ['x1', 'x2', 'x3']:
    corr, _ = pearsonr(data[col], data['y'])
    print(f'{col}-y: {corr:.4f}')

#### Feature-feature Pearson correlation coefficients

In [ ]:
# Calculate Pearson correlation coefficients between each pair of features
for col1, col2 in combinations(['x1', 'x2', 'x3'], 2):
    corr, _ = pearsonr(data[col1], data[col2])
    print(f'{col1}-{col2}: {corr:.4f}')

#### Multiple linear regression R²

In [ ]:
# Fit linear regression model and calculate R² score
model = LinearRegression()
model.fit(data.drop(columns=['y']).values, data['y'].values)
r2 = model.score(data.drop(columns=['y']).values, data['y'].values)

print(f'Model R²: {r2:.4f}')

## 2. Optimiztion

### 2.1. Objective function

In [ ]:
def feature_interactions(x1, x2, x3, y):
    '''Maximize decision tree regression R² while minimizing individual 
    feature-label and feature-feature correlations.
    
    This creates datasets where features interact to predict the label,
    but individual features alone are poor predictors, demonstrating the
    importance of feature interactions.
    
    Args:
        x1, x2, x3: Feature variables (array-like)
        y: Label variable (array-like)
    
    Returns:
        Tuple of (metrics_dict, objective_value)
    '''
    
    # Convert to numpy arrays
    x1 = np.asarray(x1)
    x2 = np.asarray(x2)
    x3 = np.asarray(x3)
    x = np.column_stack([x1, x2, x3])

    y = np.asarray(y)
    
    # Calculate individual Pearson correlations with label
    x1_y_r, _ = pearsonr(x1, y)
    x2_y_r, _ = pearsonr(x2, y)
    x3_y_r, _ = pearsonr(x3, y)
    
    # Average R² of individual features (want this LOW)
    avg_individual_r = np.mean([x1_y_r**2, x2_y_r**2, x3_y_r**2])

    # Calculate pairwise feature correlations
    x1_x2_r, _ = pearsonr(x1, x2)
    x1_x3_r, _ = pearsonr(x1, x3)
    x2_x3_r, _ = pearsonr(x2, x3)

    # Average R² of feature-feature correlations (want this LOW)
    avg_feature_r = np.mean([x1_x2_r**2, x1_x3_r**2, x2_x3_r**2])

    # Train and score the model
    model = RandomForestRegressor().fit(x, y)
    r2 = model.score(x, y)
    
    # Calculate objective
    objective = r2 - (avg_individual_r * 0.25) - (avg_feature_r * 0.25)
    
    # Compile metrics
    metrics = {
        'Model R²': float(r2),
        'Avg individual R²': float(avg_individual_r),
        'Avg feature R²': float(avg_feature_r),
    }
    
    return metrics, float(objective)

### 2.2. Hill climbing run

In [ ]:
if run_optimization:

    # Create HillClimber instance with replica exchange
    climber = HillClimber(
        data=data,
        objective_func=feature_interactions,
        initial_step_spread=initial_step_spread,
        final_step_spread=final_step_spread,
        step_spread_scheme=step_spread_scheme,
        T_max_initial=T_max_initial,
        T_min_initial=T_min_initial,
        T_max_final=T_max_final,
        T_min_final=T_min_final,
        exchange_interval=exchange_interval,
        max_time=max_time,
        n_replicas=n_replicas,
        db_path=db_path,
        checkpoint_file=output_file
    )

    # Run optimization
    best_data_df = climber.climb()

## 3. Results

### 3.1. Get replicas

In [ ]:
# If we just re-ran the optimization, collect replicas from the climber
if run_optimization:
    replicas = climber.get_replicas()

# Otherwise, load replicas from saved results
else:

    # Load the results checkpoint
    try:
        with open(output_file, 'rb') as f:
            results = pickle.load(f)

        # Get the winning replica's best data
        best_replica = max(results['replicas'], key=lambda r: r['best_objective'])

        best_data_df = pd.DataFrame(
            best_replica['best_data'],
            columns=data.columns
        )

        # Collect best data from each replica
        replicas = []

        for replica_data in results['replicas']:

            replica_df = pd.DataFrame(
                replica_data['best_data'],
                columns=data.columns
            )

            replicas.append(replica_df)

    except FileNotFoundError:

        print('Error: checkpoint file not found.')    
        results_df = pd.DataFrame()

### 3.2. Regression model

In [ ]:
# Train-test split the wining data
train_df, test_df = train_test_split(best_data_df, random_state=315)
train_df.head()

In [ ]:
# Fit regression model
model = RandomForestRegressor()
model.fit(train_df.drop(columns=['y']).values, train_df['y'].values)

# Calculate the model's R²
r2 = model.score(test_df.drop(columns=['y']).values, test_df['y'].values)

print(f'Model R²: {r2:.2f}')

In [ ]:
# Generate test set predictions for plotting
y_pred = model.predict(test_df[['x1', 'x2', 'x3']].values)
residuals = test_df['y'].values - y_pred
standardized_residuals = (residuals - np.mean(residuals)) / np.std(residuals)
root_standardized_residuals = np.sqrt(np.abs(standardized_residuals))

In [ ]:
# Regression model performance plots
fig, axs = plt.subplots(1, 3, figsize=(10, 3.5))
fig.suptitle('Winning replica: gradient boosting regression', fontsize=16)

# Plot predicted vs true values
axs[0].set_title('Predictions vs labels')
axs[0].scatter(test_df['y'].values, y_pred, s=1, c='black')
axs[0].set_xlabel('Label (y)')
axs[0].set_ylabel('Predicted y')
axs[0].set_box_aspect(1)

# Annotate the predicted vs true values plot with model R²
axs[0].annotate(
    xy=(-0.95, np.max(y_pred)),
    text=f'R²={r2:.2f}',
    ha='left', va='top',
    bbox=dict(boxstyle='square', fc='white', ec='white')
)

# Plot predicted y vs fit residuals
axs[1].set_title('Residuals')
axs[1].scatter(y_pred, residuals, s=1, c='black')
axs[1].set_xlabel('Predicted y')
axs[1].set_ylabel('Residual (y - predicted y)')
axs[1].set_box_aspect(1)

# Plot predicted y vs square root of standardized residuals
axs[2].set_title('Spread-location')
axs[2].scatter(y_pred, root_standardized_residuals, s=1, c='black')
axs[2].set_xlabel('Predicted y')
axs[2].set_ylabel('sqrt(standardized residuals)')
axs[1].set_box_aspect(1)

plt.tight_layout()
plt.show()

### 3.3. Feature-label correlations

In [ ]:
# Plot each individual feature's correlation with the label
fig, axs = plt.subplots(1, 3, figsize=(10, 3.5))
fig.suptitle('Winning replica: feature-label correlations', fontsize=16)
fig.supxlabel('Feature value')
fig.supylabel('Label (y)')

# Loop on the features
for i, feature in enumerate(['x1', 'x2', 'x3']):

    # Fit simple linear regression model
    r_value, p_value = pearsonr(
        best_data_df[feature],
        best_data_df['y']
    )

    # Draw the plot
    axs[i].set_title(feature)
    axs[i].scatter(best_data_df[feature], best_data_df['y'], s=1, c='black')
    axs[i].set_aspect('equal')

    # Format fit statistics for plot
    stats=f'R²={r_value**2:.2f}\nPearson r={r_value:.2f}\np-value={p_value:.3f}'

    # Annotate plot with fit statistics
    axs[i].annotate(
        xy=(-0.95, 0.95),
        text=stats,
        ha='left', va='top',
        bbox=dict(boxstyle='square', fc='white', ec='white')
    )

plt.tight_layout()
plt.show()

### 3.4. Feature-feature correlations

In [ ]:
# Plot correlation between each pair of features
fig, axs = plt.subplots(1, 3, figsize=(10, 3.5))
fig.suptitle('Winning replica: feature correlations', fontsize=16)

# Loop on feature pairs
for i, features in enumerate([['x1', 'x2'],['x1', 'x3'], ['x2', 'x3']]):

    # Fit simple linear regression model
    r_value, p_value = pearsonr(
        best_data_df[features[0]],
        best_data_df[features[1]]
    )

    # Draw the plot
    axs[i].set_title(f'{features[0]} & {features[1]}')
    axs[i].scatter(best_data_df[features[0]], best_data_df[features[1]], s=1, c='black')
    axs[i].set_xlabel(features[0])
    axs[i].set_ylabel(features[1])
    axs[i].set_aspect('equal')

    # format fit statistics for plotting
    stats=f'R²={r_value**2:.2f}\nPearson r={r_value:.2f}\np-value={p_value:.3f}'

    # Annotate the plot with the fit statistics
    axs[i].annotate(
        xy=(-0.95, 0.95),
        text=stats,
        ha='left', va='top', 
        bbox=dict(boxstyle='square', fc='white', ec='white')
    )

plt.tight_layout()
plt.show()

### 3.5. Replicate regression models

In [ ]:
# Set plot dimensions
ncols = 5
nrows = len(replicas) // ncols

if len(replicas) % ncols != 0:
    nrows+=1

fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols*2, nrows*2))
axs = axs.flatten()

fig.suptitle('Replica regression models', fontsize=14)
fig.supxlabel('Label (y)', fontsize=12)
fig.supylabel('predicted y', fontsize=12)

# Loop on the replicates
for i, replica in enumerate(replicas):

    # Split replicate data into training and testing sets
    replicate_train_df, replicate_test_df = train_test_split(replica, random_state=315)

    # Fit regression model
    model = RandomForestRegressor()
    model.fit(replicate_train_df.drop(columns=['y']).values, replicate_train_df['y'].values)

    # Generate test set predictions for plotting
    y_pred = model.predict(replicate_test_df.drop(columns=['y']).values)

    # Calculate model R²
    r2 = model.score(
        replicate_test_df.drop(columns=['y']).values,
        replicate_test_df['y'].values
    )

    # Draw the plot
    axs[i].set_title(f'R²={r2:.2f}', fontsize=10)
    axs[i].scatter(replicate_test_df['y'], y_pred, s=0.1, c='black')

plt.tight_layout()
plt.show()